# Actividad 3 | Aprendizaje supervisado y no supervisado

**Dataset:** *eCommerce behavior data from multi category store* — Kaggle  
**Fuente:** `mkechinov/ecommerce-behavior-data-from-multi-category-store`  
  
- Antonio Ramón Valerio Tejada — A01797448  


**Herramienta principal:** PySpark MLlib  


## 1. Introducción teórica

El aprendizaje automático permite identificar patrones en los datos para realizar predicciones o encontrar estructuras ocultas. De forma general, se divide en aprendizaje supervisado y aprendizaje no supervisado.

En el aprendizaje supervisado, los datos incluyen una variable objetivo o etiqueta. El modelo aprende la relación entre las variables de entrada y esa etiqueta para predecir nuevos casos.

En el aprendizaje no supervisado, los datos no tienen una etiqueta definida. El objetivo es descubrir patrones, similitudes o grupos naturales dentro de la información. Algunos algoritmos comunes incluyen clustering (K-Means) y análisis de asociación.

Antes de entrenar los modelos, es necesario construir una muestra representativa, limpiar inconsistencias, transformar variables y dividir los datos en entrenamiento y prueba. En Big Data, esto requiere estrategias específicas para trabajar con volúmenes grandes de información.


## 2. Instalación e importación de librerías

In [ ]:
# Ejecutar solo si tu entorno no tiene estas librerías.
# !pip install pyspark findspark kagglehub pandas matplotlib

In [ ]:
import os
import findspark
findspark.init()

import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator

SEED = 42

## 3. Creación de SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Actividad4_Supervisado_NoSupervisado_eCommerce")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

# Configuración para visualizar mejor los DataFrames en notebooks.
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark


## 4. Selección de los datos

Para evitar tiempos de procesamiento altos, se construirá una muestra inicial a partir del dataset global.

Estrategia:

1. Descargar o localizar el dataset de Kaggle.
2. Leer únicamente un archivo mensual, preferentemente `2019-Oct.csv`.
3. Tomar una fracción pequeña del dataset completo para trabajar localmente.
4. Generar variables de caracterización.
5. Crear particiones por tipo de evento y segmento de precio.
6. Recuperar un número limitado de instancias por partición para formar la muestra inicial **M**.

In [ ]:
# Descargar el dataset desde Kaggle Hub
import kagglehub

DATASET_DIR = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")
print("Dataset descargado en:", DATASET_DIR)

csv_path = None
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file == "2019-Oct.csv":
            csv_path = os.path.join(root, file)

if csv_path is None:
    raise FileNotFoundError("No se encontró 2019-Oct.csv. Revisa la ruta del dataset.")

print("Archivo seleccionado:", csv_path)

Dataset descargado en: C:\\Users\\YOLOA\\.cache\\kagglehub\\datasets\\mkechinov\\ecommerce-behavior-data-from-multi-category-store\\versions\\8
Archivo seleccionado: C:\\Users\\YOLOA\\.cache\\kagglehub\\datasets\\mkechinov\\ecommerce-behavior-data-from-multi-category-store\\versions\\8\\2019-Oct.csv


In [ ]:
# Cargar el archivo CSV en un DataFrame de PySpark
df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)

print("Columnas originales:")
print(df_raw.columns)

df_raw.printSchema()
df_raw.show(5, truncate=False)

Columnas originales:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+-------------------------

In [ ]:
# Muestra base para evitar errores de memoria en Spark local.
BASE_SAMPLE_FRACTION = 0.01

df = df_raw.sample(withReplacement=False, fraction=BASE_SAMPLE_FRACTION, seed=SEED)

print("Muestra base creada.")
df.show(5, truncate=False)

Muestra base creada.
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+
|2019-10-01 00:03:38|view      |5300650   |2053013563173241677|NULL                  |vitek  |12.84 |552584835|b56e1552-5a77-44bc-af49-d1b1ecd00b4e|
|2019-10-01 00:04:45|view      |1004870   |2053013555631882655|electronics.smartphone|samsung|286.86|513881159|387ba75c-47bb-480a-9f86-b7c5cc57cf6b|
|2019-10-01 00:06:19|view      |1004246   |2053013555631882655|electronics.smartphone|apple  |736.18|533940457|309e5dba-628b-491a-98ec-4c2906db831a|
|2019-10-01 00:11:38|view      |9300040   |2053013554524586339|NULL                  

### 4.1 Limpieza inicial y variables de caracterización

Se eliminan registros con nulos en columnas esenciales, se corrigen tipos de datos y se generan variables derivadas:

- `event_date`: fecha del evento.
- `hour`: hora del día.
- `day_of_week`: día de la semana.
- `category_level_1`: categoría principal.
- `price_segment`: segmento de precio definido con cuantiles aproximados.
- `is_purchase`: variable objetivo binaria para el modelo supervisado.

In [ ]:
# Eliminar registros con valores nulos en campos críticos
# y generar variables derivadas para el análisis
df_clean = (
    df
    .dropna(subset=["event_time", "event_type", "product_id", "price", "user_id", "user_session"])
    .filter(F.col("price") >= 0)
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.date_format("event_time", "E"))
    .withColumn(
        "category_level_1",
        F.when(F.col("category_code").isNull(), "unknown")
         .otherwise(F.split(F.col("category_code"), "\\.").getItem(0))
    )
    .withColumn("brand", F.when(F.col("brand").isNull(), "unknown").otherwise(F.col("brand")))
)

q25, q50, q75 = df_clean.approxQuantile("price", [0.25, 0.50, 0.75], 0.05)

df_clean = (
    df_clean
    .withColumn(
        "price_segment",
        F.when(F.col("price") <= q25, "bajo")
         .when(F.col("price") <= q50, "medio_bajo")
         .when(F.col("price") <= q75, "medio_alto")
         .otherwise("alto")
    )
    .withColumn("is_purchase", F.when(F.col("event_type") == "purchase", 1.0).otherwise(0.0))
)

df_clean.select(
    "event_time", "event_type", "price", "hour",
    "day_of_week", "category_level_1", "price_segment", "is_purchase"
).show(10, truncate=False)

+-------------------+----------+------+----+-----------+----------------+-------------+-----------+
|event_time         |event_type|price |hour|day_of_week|category_level_1|price_segment|is_purchase|
+-------------------+----------+------+----+-----------+----------------+-------------+-----------+
|2019-10-01 00:03:38|view      |12.84 |0   |Tue        |unknown         |bajo         |0.0        |
|2019-10-01 00:04:45|view      |286.86|0   |Tue        |electronics     |medio_alto   |0.0        |
|2019-10-01 00:06:19|view      |736.18|0   |Tue        |electronics     |alto         |0.0        |
|2019-10-01 00:11:38|view      |334.12|0   |Tue        |unknown         |alto         |0.0        |
|2019-10-01 01:45:37|view      |177.47|1   |Tue        |electronics     |medio_alto   |0.0        |
|2019-10-01 01:58:12|view      |241.89|1   |Tue        |appliances      |medio_alto   |0.0        |
|2019-10-01 02:00:31|view      |615.19|2   |Tue        |unknown         |alto         |0.0        |


In [ ]:
# Obtener estadísticas generales de la muestra
df_clean.select(
    F.count("*").alias("registros"),
    F.mean("price").alias("precio_promedio"),
    F.min("price").alias("precio_minimo"),
    F.max("price").alias("precio_maximo")
).show()

print("Distribución de event_type:")
df_clean.groupBy("event_type").count().orderBy(F.desc("count")).show(truncate=False)

print("Distribución de price_segment:")
df_clean.groupBy("price_segment").count().orderBy(F.desc("count")).show(truncate=False)

print("Distribución de category_level_1:")
df_clean.groupBy("category_level_1").count().orderBy(F.desc("count")).show(10, truncate=False)

+---------+------------------+-------------+-------------+
|registros|   precio_promedio|precio_minimo|precio_maximo|
+---------+------------------+-------------+-------------+
|   425322|291.56316019392347|          0.0|      2574.07|
+---------+------------------+-------------+-------------+

Distribución de event_type:
+----------+------+
|event_type|count |
+----------+------+
|view      |408827|
|cart      |9110  |
|purchase  |7385  |
+----------+------+

Distribución de price_segment:
+-------------+------+
|price_segment|count |
+-------------+------+
|alto         |120992|
|bajo         |107066|
|medio_alto   |101094|
|medio_bajo   |96170 |
+-------------+------+

Distribución de category_level_1:
+----------------+------+
|category_level_1|count |
+----------------+------+
|electronics     |162028|
|unknown         |135517|
|appliances      |49603 |
|computers       |23468 |
|apparel         |15331 |
|furniture       |12465 |
|auto            |9929  |
|construction    |7310  |

### Revisión de valores atípicos

Para identificar posibles valores atípicos, se revisó principalmente la variable 'price', ya que es la variable numérica con mayor impacto en los modelos. Durante la limpieza se eliminaron valores negativos que no tenían sentido en el contexto de precios de productos.

Además, se creó la variable 'price_segment' mediante cuantiles, lo que permite reducir el efecto de la dispersión del precio y analizar los productos por rangos en lugar de depender únicamente del valor numérico exacto.

### 4.2 Generación de la muestra de trabajo

Para construir la muestra M, se utiliza una selección controlada de registros considerando dos variables relevantes del comportamiento de compra:

* event_type
* price_segment

La combinación de estas variables permite incluir distintos tipos de interacción del usuario y diferentes rangos de precio dentro de la muestra. Dado que los eventos no aparecen con la misma frecuencia, se aplica muestreo estratificado para asegurar representación de todos los grupos.

Esta muestra no pretende replicar exactamente la distribución completa de la base original, sino generar un subconjunto equilibrado y manejable que permita probar los algoritmos de aprendizaje automático de forma efectiva.


In [ ]:
# Crear una etiqueta que combine el tipo de evento y el segmento de precio
df_part = (
    df_clean
    .withColumn(
        "partition_rule",
        F.concat_ws("_", F.col("event_type"), F.col("price_segment"))
    )
)

partition_counts = (
    df_part
    .groupBy("partition_rule")
    .count()
    .orderBy(F.desc("count"))
)

partition_counts.show(50, truncate=False)

+-------------------+------+
|partition_rule     |count |
+-------------------+------+
|view_alto          |115937|
|view_bajo          |104225|
|view_medio_alto    |96090 |
|view_medio_bajo    |92575 |
|cart_medio_alto    |2977  |
|cart_alto          |2861  |
|purchase_alto      |2194  |
|purchase_medio_alto|2027  |
|cart_medio_bajo    |1997  |
|purchase_medio_bajo|1598  |
|purchase_bajo      |1566  |
|cart_bajo          |1275  |
+-------------------+------+



In [ ]:
#Construcción de la muestra M
N_PER_PARTITION = 600

allocation = (
    partition_counts
    .withColumn(
        "fraccion_muestreo",
        F.when(F.col("count") <= N_PER_PARTITION, F.lit(1.0))
         .otherwise(F.lit(N_PER_PARTITION) / F.col("count"))
    )
)

fractions = {
    row["partition_rule"]: min(float(row["fraccion_muestreo"]), 1.0)
    for row in allocation.collect()
}

M = df_part.sampleBy("partition_rule", fractions=fractions, seed=SEED)

print("Muestra M creada.")
print(f"Tamaño aproximado de M: {M.count():,}")

print("Distribución de la muestra M por partición:")
M.groupBy("partition_rule").count().orderBy(F.desc("count")).show(50, truncate=False)

print("Distribución de la variable objetivo en M:")
M.groupBy("is_purchase").count().orderBy("is_purchase").show()


Muestra M creada.
Tamaño aproximado de M: 7,298
Distribución de la muestra M por partición:
+-------------------+-----+
|partition_rule     |count|
+-------------------+-----+
|view_medio_alto    |656  |
|cart_medio_alto    |642  |
|view_bajo          |631  |
|cart_bajo          |619  |
|purchase_medio_bajo|619  |
|cart_alto          |618  |
|purchase_medio_alto|601  |
|purchase_bajo      |593  |
|cart_medio_bajo    |582  |
|view_alto          |582  |
|purchase_alto      |581  |
|view_medio_bajo    |574  |
+-------------------+-----+

Distribución de la variable objetivo en M:
+-----------+-----+
|is_purchase|count|
+-----------+-----+
|        0.0| 4904|
|        1.0| 2394|
+-----------+-----+



## 5. Preparación de la muestra para modelado

Antes de aplicar los algoritmos de aprendizaje automático, fue necesario realizar una serie de transformaciones para asegurar que los datos pudieran ser procesados correctamente por las bibliotecas de machine learning.

Las principales actividades realizadas fueron:

* Eliminación de registros con información incompleta o inconsistente.
* Tratamiento de valores faltantes en variables categóricas.
* Codificación de variables categóricas a formato numérico mediante técnicas de indexación y codificación binaria.
* Integración de las variables seleccionadas en un único vector de características (features).
* Definición de la variable objetivo is_purchase, utilizada para identificar si una interacción terminó en compra o no.

Como resultado, se obtuvo un conjunto de datos estructurado y listo para la construcción de modelos de aprendizaje supervisado y no supervisado.


In [ ]:
# Selección y preparación de variables predictoras
model_cols = [
    "price",
    "hour",
    "event_type",
    "category_level_1",
    "brand",
    "price_segment",
    "day_of_week",
    "is_purchase",
    "partition_rule"
]

M_model = (
    M
    .select(*model_cols)
    .dropna(subset=["price", "hour", "is_purchase"])
)

# Limitar marcas para evitar demasiadas categorías.
top_brands = [
    row["brand"]
    for row in M_model.groupBy("brand").count().orderBy(F.desc("count")).limit(25).collect()
]

M_model = M_model.withColumn(
    "brand_limited",
    F.when(F.col("brand").isin(top_brands), F.col("brand")).otherwise("other")
)

M_model.select(
    "price", "hour", "event_type", "category_level_1",
    "brand_limited", "price_segment", "day_of_week", "is_purchase"
).show(10, truncate=False)

+-------+----+----------+----------------+-------------+-------------+-----------+-----------+
|price  |hour|event_type|category_level_1|brand_limited|price_segment|day_of_week|is_purchase|
+-------+----+----------+----------------+-------------+-------------+-----------+-----------+
|65.38  |2   |view      |unknown         |other        |bajo         |Tue        |0.0        |
|130.76 |3   |view      |electronics     |samsung      |medio_bajo   |Tue        |0.0        |
|56.23  |3   |purchase  |appliances      |redmond      |bajo         |Tue        |1.0        |
|39.9   |3   |cart      |unknown         |other        |bajo         |Tue        |0.0        |
|82.29  |3   |purchase  |auto            |other        |medio_bajo   |Tue        |1.0        |
|1415.48|3   |cart      |electronics     |apple        |alto         |Tue        |0.0        |
|17.99  |3   |purchase  |appliances      |other        |bajo         |Tue        |1.0        |
|391.0  |3   |purchase  |unknown         |sony    

In [ ]:
#Distribución de la variable objetivo
target_distribution = (
    M_model
    .groupBy("is_purchase")
    .count()
    .withColumn("porcentaje", F.round(F.col("count") / F.sum("count").over(Window.partitionBy()) * 100, 2))
)

target_distribution.show()

+-----------+-----+----------+
|is_purchase|count|porcentaje|
+-----------+-----+----------+
|        1.0| 2394|      32.8|
|        0.0| 4904|      67.2|
+-----------+-----+----------+



### 5.1 Consideraciones sobre la muestra utilizada

La muestra M se generó mediante una selección controlada por grupos, con el propósito de incluir suficientes registros de eventos menos frecuentes, como las compras. Debido a esto, la proporción de compras en M es mayor que la observada en los datos originales.

Esta decisión resulta adecuada para fines de experimentación y entrenamiento de modelos, ya que permite trabajar con más ejemplos de la clase minoritaria. Sin embargo, los porcentajes obtenidos no reflejan la realidad del negocio y deben interpretarse con cautela al evaluar el desempeño del modelo.

## 6. División de datos para entrenamiento y prueba

Para evaluar el desempeño del modelo supervisado, la muestra se divide en dos subconjuntos: entrenamiento y prueba. Se utiliza una proporción 80/20, donde el 80 porciento de los datos se emplean para entrenar el modelo y el 20 porciento restante para evaluarlo.

La separación se realiza de forma estratificada tomando como referencia la variable is_purchase. De esta manera, tanto el conjunto de entrenamiento como el de prueba conservan una distribución similar de la variable objetivo.

Adicionalmente, se calculan pesos de clase a partir del conjunto de entrenamiento. Esto ayuda a compensar el desbalance entre clases y evita que el modelo favorezca excesivamente la clase mayoritaria.

In [ ]:
# División estratificada 80/20 usando la variable objetivo is_purchase.
# Esto ayuda a mantener una proporción similar de compras y no compras en train y test.

M_indexed = M_model.withColumn("row_id", F.monotonically_increasing_id())

train_fraction_by_class = {
    0.0: 0.80,
    1.0: 0.80
}

train = M_indexed.sampleBy(
    "is_purchase",
    fractions=train_fraction_by_class,
    seed=SEED
)

train_ids = train.select("row_id").withColumn("in_train", F.lit(1))

test = (
    M_indexed
    .join(train_ids, on="row_id", how="left")
    .filter(F.col("in_train").isNull())
    .drop("in_train")
)

print(f"Train: {train.count():,}")
print(f"Test: {test.count():,}")

print("Distribución en entrenamiento:")
train.groupBy("is_purchase").count().orderBy("is_purchase").show()

print("Distribución en prueba:")
test.groupBy("is_purchase").count().orderBy("is_purchase").show()

# Cálculo de pesos de clase a partir del conjunto de entrenamiento.
# Esto es importante porque, sin pesos, el modelo puede favorecer la clase mayoritaria.
class_counts = {
    row["is_purchase"]: row["count"]
    for row in train.groupBy("is_purchase").count().collect()
}

n_train = sum(class_counts.values())

weight_0 = n_train / (2 * class_counts.get(0.0, 1))
weight_1 = n_train / (2 * class_counts.get(1.0, 1))

train = train.withColumn(
    "class_weight",
    F.when(F.col("is_purchase") == 1.0, F.lit(weight_1)).otherwise(F.lit(weight_0))
)

test = test.withColumn(
    "class_weight",
    F.when(F.col("is_purchase") == 1.0, F.lit(weight_1)).otherwise(F.lit(weight_0))
)

print(f"Peso clase 0: {weight_0:.4f}")
print(f"Peso clase 1: {weight_1:.4f}")


Train: 5,838
Test: 1,460
Distribución en entrenamiento:
+-----------+-----+
|is_purchase|count|
+-----------+-----+
|        0.0| 3924|
|        1.0| 1914|
+-----------+-----+

Distribución en prueba:
+-----------+-----+
|is_purchase|count|
+-----------+-----+
|        0.0|  980|
|        1.0|  480|
+-----------+-----+

Peso clase 0: 0.7439
Peso clase 1: 1.5251


## 7. Implementación del modelo supervisado

Para la tarea de clasificación se seleccionó el algoritmo Random Forest Classifier, cuyo objetivo es predecir si una interacción de usuario termina en una compra (is_purchase = 1) o no (is_purchase = 0).

Durante la construcción del modelo se excluyó la variable event_type del conjunto de predictores, ya que la variable objetivo fue generada a partir de dicha información. Esto permite evitar un sesgo perfecto en las predicciones.

Adicionalmente, se incorporaron pesos de clase con el fin de reducir el impacto del desbalance existente entre compras y no compras, favoreciendo una evaluación más equilibrada del desempeño.


In [ ]:
### Entrenamiento del modelo Random Forest
categorical_cols = [
    "category_level_1",
    "brand_limited",
    "price_segment",
    "day_of_week"
]

numeric_cols = [
    "price",
    "hour"
]

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{col}_idx" for col in categorical_cols],
    outputCols=[f"{col}_ohe" for col in categorical_cols],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{col}_ohe" for col in categorical_cols],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    labelCol="is_purchase",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=80,
    maxDepth=7,
    minInstancesPerNode=5,
    seed=SEED
)

rf_pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])

rf_model = rf_pipeline.fit(train)
predictions = rf_model.transform(test)

predictions.select(
    "is_purchase", "prediction", "probability",
    "price", "hour", "category_level_1", "brand_limited", "price_segment"
).show(20, truncate=False)


+-----------+----------+----------------------------------------+------+----+----------------+-------------+-------------+
|is_purchase|prediction|probability                             |price |hour|category_level_1|brand_limited|price_segment|
+-----------+----------+----------------------------------------+------+----+----------------+-------------+-------------+
|1.0        |0.0       |[0.5294815774398637,0.4705184225601363] |56.23 |3   |appliances      |redmond      |bajo         |
|1.0        |0.0       |[0.5131821319198343,0.4868178680801657] |17.99 |3   |appliances      |other        |bajo         |
|0.0        |0.0       |[0.5120844110570075,0.4879155889429924] |19.82 |3   |unknown         |vitek        |bajo         |
|1.0        |0.0       |[0.5146858154047722,0.4853141845952277] |476.18|4   |appliances      |samsung      |alto         |
|0.0        |1.0       |[0.47875982684552226,0.5212401731544778]|174.75|4   |electronics     |samsung      |medio_alto   |
|1.0        |1.0